# Sistem Rekomendasi Channel YouTube Menggunakan IndoBERT
## Penerapan IndoBERT untuk Mengukur Kemiripan Berdasarkan Judul Konten pada Sistem Rekomendasi Channel YouTube

**Metodologi:** Content-Based Filtering menggunakan IndoBERT + Cosine Similarity

**Alur Pipeline:**
1. Load Dataset JSON
2. Text Cleaning
3. Case Folding
4. Preprocessing dengan Stanza (Tokenisasi)
5. Load Model IndoBERT
6. Generate Embedding Judul Video
7. Agregasi Vector per Channel (Mean Pooling)
8. Cosine Similarity Matrix
9. Fungsi Rekomendasi
10. Evaluasi (Precision@K)

---
## 1. Install & Import Library

In [1]:
# Install library yang diperlukan
# Jalankan cell ini sekali (terutama di Google Colab)
%pip install torch transformers stanza scikit-learn pandas numpy tqdm --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ============================================================
# BAGIAN 1: IMPORT LIBRARY
# ============================================================

import json
import re
import os
import numpy as np
import pandas as pd

# Deep Learning & NLP
import torch
from transformers import AutoTokenizer, AutoModel

# Stanza untuk Tokenisasi Bahasa Indonesia
import stanza

# Similarity & Evaluation
from sklearn.metrics.pairwise import cosine_similarity

# Progress bar
from tqdm import tqdm

# Konfigurasi device (GPU jika tersedia, fallback ke CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Menggunakan device  : {device}')
print(f'PyTorch version     : {torch.__version__}')
print('Import library selesai.')

c:\Users\ACER\.conda\envs\model_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Menggunakan device  : cpu
PyTorch version     : 2.10.0+cpu
Import library selesai.


---
## 2. Load Dataset JSON

In [3]:
# ============================================================
# BAGIAN 2: LOAD DATASET JSON
# ============================================================

# Path file dataset
# Jika di Google Colab: upload data_video.json atau mount Google Drive
# lalu sesuaikan path di bawah ini.
DATA_PATH = 'data_video.json'

# Load JSON
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# Konversi ke DataFrame
df = pd.DataFrame(raw_data)

# ─── Informasi Dataset ───────────────────────────────────────
print('=' * 55)
print('           INFORMASI DATASET')
print('=' * 55)
print(f'Total record (video)   : {len(df):,}')
print(f'Total kolom            : {len(df.columns)}')
print(f'Nama kolom             : {list(df.columns)}')
print(f'Jumlah channel unik    : {df["nama_channel"].nunique()}')
print(f'Jumlah kategori unik   : {df["kategori"].nunique()}')
print('=' * 55)

# Distribusi video per channel
dist = df['nama_channel'].value_counts()
print(f'\nDistribusi jumlah video per channel:')
print(f'  Min  : {dist.min()} video')
print(f'  Max  : {dist.max()} video')
print(f'  Rata : {dist.mean():.1f} video')
print()

# Tampilkan 5 baris pertama
df.head()

           INFORMASI DATASET
Total record (video)   : 10,000
Total kolom            : 9
Nama kolom             : ['id', 'link_channel', 'nama_channel', 'kategori', 'jumlah_pelanggan', 'judul', 'link', 'jumlah_tayangan', 'tanggal_upload']
Jumlah channel unik    : 100
Jumlah kategori unik   : 10

Distribusi jumlah video per channel:
  Min  : 100 video
  Max  : 100 video
  Rata : 100.0 video



,id,link_channel,nama_channel,kategori,jumlah_pelanggan,judul,link,jumlah_tayangan,tanggal_upload
0,1,https://www.youtube.com/@GadgetIn/videos,@GadgetIn,Gadgets,13800000,Unboxing iPhone 17 Pro PALSU yang SANGAT MIRIP...,https://www.youtube.com/watch?v=zE5H9KQ_Hyg,1700000,5 days ago
1,2,https://www.youtube.com/@GadgetIn/videos,@GadgetIn,Gadgets,13800000,RAJA TERAKHIR HP SAMSUNG!,https://www.youtube.com/watch?v=snB4jbtscxU,932000,10 days ago
2,3,https://www.youtube.com/@GadgetIn/videos,@GadgetIn,Gadgets,13800000,Rp1.599 Juta! Ketika OPPO NIAT bikin HP murah...,https://www.youtube.com/watch?v=3KBJtbEAdzs,802000,11 days ago
3,4,https://www.youtube.com/@GadgetIn/videos,@GadgetIn,Gadgets,13800000,"Kalau Apple niat, iPhone bisa seworth it ini.....",https://www.youtube.com/watch?v=rkLpVyRGCPw,1300000,2 weeks ago
4,5,https://www.youtube.com/@GadgetIn/videos,@GadgetIn,Gadgets,13800000,Xiaomi pun ngeluh soal fenomena ini...,https://www.youtube.com/watch?v=Z4m_fwJ5eHQ&pp...,1200000,2 weeks ago


In [4]:
# ─── Daftar channel yang tersedia ────────────────────────────
print('Daftar Channel YouTube dalam Dataset:')
print('-' * 50)
for i, (ch, cnt) in enumerate(df['nama_channel'].value_counts().items(), 1):
    print(f'{i:>3}. {ch:<40} ({cnt} video)')

Daftar Channel YouTube dalam Dataset:
--------------------------------------------------
  1. @GadgetIn                                (100 video)
  2. @JagatReview                             (100 video)
  3. @GadgetGaul                              (100 video)
  4. @DHIARCOM                                (100 video)
  5. @DKIDchannel                             (100 video)
  6. @PricebookIndonesia                      (100 video)
  7. @Sobat_HAPE                              (100 video)
  8. @projectreview                           (100 video)
  9. @K2G                                     (100 video)
 10. @YoutuberCupu                            (100 video)
 11. @NexCarlos                               (100 video)
 12. @riasukmawijaya                          (100 video)
 13. @tanboykun                               (100 video)
 14. @KUBILER                                 (100 video)
 15. @MamankKuliner                           (100 video)
 16. @Melkibajaj                         

---
## 3. Text Cleaning

Pembersihan teks secara **minimalis**: hanya menghapus noise tanpa makna bahasa
(emoji, simbol dekoratif non-standar, karakter encoding rusak).
Tanda baca standar **(titik, koma, tanda tanya)** tetap dipertahankan karena IndoBERT memanfaatkannya.

In [5]:
# ============================================================
# BAGIAN 3: TEXT CLEANING
# ============================================================

def text_cleaning(text):
    """
    Membersihkan teks secara minimalis sesuai metodologi penelitian.

    Tahapan:
    1. Menghapus emoji dan simbol Unicode non-standar
    2. Menghapus karakter encoding yang rusak / tidak dikenali
    3. Menghapus karakter khusus / dekoratif yang bukan tanda baca standar
    4. Merapikan spasi berlebih

    TIDAK menghapus: titik, koma, tanda tanya, tanda seru, tanda hubung
    karena digunakan IndoBERT untuk memahami batas kalimat & intonasi makna.
    """
    if not isinstance(text, str):
        return ''

    # 1. Hapus emoji dan simbol Unicode non-standar
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"   # emoticons wajah
        "\U0001F300-\U0001F5FF"   # simbol & piktogram
        "\U0001F680-\U0001F6FF"   # transport & peta
        "\U0001F1E0-\U0001F1FF"   # bendera
        "\U00002600-\U000026FF"   # simbol umum
        "\U00002700-\U000027BF"   # Dingbats
        "\U0001F900-\U0001F9FF"   # simbol tambahan
        "\U0001FA00-\U0001FA6F"   # simbol tambahan-A
        "\U0001FA70-\U0001FAFF"   # simbol tambahan-B
        "\U00002300-\U000023FF"   # teknis
        "]+",
        flags=re.UNICODE
    )
    text = emoji_pattern.sub(' ', text)

    # 2. Hapus karakter non-printable / encoding rusak
    text = re.sub(r'[\x00-\x08\x0B-\x0C\x0E-\x1F\x7F]', ' ', text)

    # 3. Hapus simbol dekoratif yang bukan tanda baca standar
    #    Pertahankan: huruf, angka, spasi, . , ? ! - ( ) / @ # % + = : ; ' "
    text = re.sub(r'[^\w\s.,?!\-()/@#%+=\'":;]', ' ', text, flags=re.UNICODE)

    # 4. Rapikan spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# ─── Terapkan Text Cleaning ───────────────────────────────────
df['judul_clean'] = df['judul'].apply(text_cleaning)

# Tampilkan contoh hasil cleaning
print('Contoh Hasil Text Cleaning:')
print('=' * 70)
for _, row in df[['judul', 'judul_clean']].head(8).iterrows():
    print(f'SEBELUM : {row["judul"]}')
    print(f'SESUDAH : {row["judul_clean"]}')
    print('-' * 70)

Contoh Hasil Text Cleaning:
SEBELUM : Unboxing iPhone 17 Pro PALSU yang SANGAT MIRIP ASLINYA...
SESUDAH : Unboxing iPhone 17 Pro PALSU yang SANGAT MIRIP ASLINYA...
----------------------------------------------------------------------
SEBELUM : RAJA TERAKHIR HP SAMSUNG!
SESUDAH : RAJA TERAKHIR HP SAMSUNG!
----------------------------------------------------------------------
SEBELUM : Rp1.599 Juta! Ketika OPPO NIAT bikin HP murah...
SESUDAH : Rp1.599 Juta! Ketika OPPO NIAT bikin HP murah...
----------------------------------------------------------------------
SEBELUM : Kalau Apple niat, iPhone bisa seworth it ini... - Review iPhone 17
SESUDAH : Kalau Apple niat, iPhone bisa seworth it ini... - Review iPhone 17
----------------------------------------------------------------------
SEBELUM : Xiaomi pun ngeluh soal fenomena ini...
SESUDAH : Xiaomi pun ngeluh soal fenomena ini...
----------------------------------------------------------------------
SEBELUM : Rekomendasi HP TERBAIK buat A

---
## 4. Case Folding

Mengubah seluruh teks menjadi huruf kecil (**lowercase**) agar sesuai dengan
model **IndoBERT Base Uncased** yang dilatih pada teks huruf kecil.

In [6]:
# ============================================================
# BAGIAN 4: CASE FOLDING
# ============================================================

def case_folding(text):
    """
    Mengubah seluruh karakter huruf menjadi huruf kecil (lowercase).
    Sesuai dengan IndoBERT Base Uncased yang tidak membedakan kapitalisasi.
    """
    if not isinstance(text, str):
        return ''
    return text.lower()


# Terapkan case folding setelah text cleaning
df['judul_lower'] = df['judul_clean'].apply(case_folding)

# Tampilkan contoh
print('Contoh Hasil Case Folding (setelah Text Cleaning):')
print('=' * 70)
for _, row in df[['judul_clean', 'judul_lower']].head(5).iterrows():
    print(f'SEBELUM : {row["judul_clean"]}')
    print(f'SESUDAH : {row["judul_lower"]}')
    print('-' * 70)

Contoh Hasil Case Folding (setelah Text Cleaning):
SEBELUM : Unboxing iPhone 17 Pro PALSU yang SANGAT MIRIP ASLINYA...
SESUDAH : unboxing iphone 17 pro palsu yang sangat mirip aslinya...
----------------------------------------------------------------------
SEBELUM : RAJA TERAKHIR HP SAMSUNG!
SESUDAH : raja terakhir hp samsung!
----------------------------------------------------------------------
SEBELUM : Rp1.599 Juta! Ketika OPPO NIAT bikin HP murah...
SESUDAH : rp1.599 juta! ketika oppo niat bikin hp murah...
----------------------------------------------------------------------
SEBELUM : Kalau Apple niat, iPhone bisa seworth it ini... - Review iPhone 17
SESUDAH : kalau apple niat, iphone bisa seworth it ini... - review iphone 17
----------------------------------------------------------------------
SEBELUM : Xiaomi pun ngeluh soal fenomena ini...
SESUDAH : xiaomi pun ngeluh soal fenomena ini...
----------------------------------------------------------------------


---
## 5. Tokenisasi dengan Stanza (Bahasa Indonesia)

Stanza digunakan untuk **tokenisasi** teks Bahasa Indonesia.
Output berupa teks yang sudah dinormalisasi (token yang bergabung kembali).

In [7]:
# ============================================================
# BAGIAN 5: TOKENISASI DENGAN STANZA
# ============================================================

# ─── Langkah 1: Download model Bahasa Indonesia ──────────────
# Hanya perlu diunduh sekali; jika sudah ada, akan di-skip otomatis
stanza.download('id', verbose=False)   # 'id' = kode bahasa Indonesia
print('Model Stanza Bahasa Indonesia siap.')

Model Stanza Bahasa Indonesia siap.


In [8]:
# ─── Langkah 2: Inisialisasi pipeline Stanza ─────────────────
# Hanya processor 'tokenize' yang diaktifkan untuk efisiensi
nlp_stanza = stanza.Pipeline(
    lang='id',
    processors='tokenize',
    verbose=False
)


def tokenize_stanza(text):
    """
    Melakukan tokenisasi teks menggunakan Stanza Bahasa Indonesia.

    Output adalah teks yang dinormalisasi: token-token dipisahkan spasi.
    Representasi ini digunakan sebagai input ke tokenizer IndoBERT.
    """
    if not text or not text.strip():
        return text

    doc = nlp_stanza(text)
    # Gabungkan semua token dari semua kalimat
    tokens = []
    for sentence in doc.sentences:
        for token in sentence.tokens:
            tokens.append(token.text)

    return ' '.join(tokens)


# ─── Langkah 3: Terapkan tokenisasi ke seluruh dataset ───────
print(f'Memproses tokenisasi Stanza untuk {len(df):,} judul...')
print('(Proses ini memerlukan beberapa menit)\n')

tqdm.pandas(desc='Tokenisasi Stanza')
df['judul_tokenized'] = df['judul_lower'].progress_apply(tokenize_stanza)

print('\nTokenisasi selesai!')
print('\nContoh hasil tokenisasi:')
print('=' * 70)
for _, row in df[['judul_lower', 'judul_tokenized']].head(5).iterrows():
    print(f'INPUT  : {row["judul_lower"]}')
    print(f'OUTPUT : {row["judul_tokenized"]}')
    print('-' * 70)

Memproses tokenisasi Stanza untuk 10,000 judul...
(Proses ini memerlukan beberapa menit)



Tokenisasi Stanza: 100%|██████████| 10000/10000 [02:27<00:00, 67.59it/s] 


Tokenisasi selesai!

Contoh hasil tokenisasi:
INPUT  : unboxing iphone 17 pro palsu yang sangat mirip aslinya...
OUTPUT : unboxing iphone 17 pro palsu yang sangat mirip aslinya . . .
----------------------------------------------------------------------
INPUT  : raja terakhir hp samsung!
OUTPUT : raja terakhir hp samsung !
----------------------------------------------------------------------
INPUT  : rp1.599 juta! ketika oppo niat bikin hp murah...
OUTPUT : rp1.599 juta ! ketika oppo niat bikin hp murah . . .
----------------------------------------------------------------------
INPUT  : kalau apple niat, iphone bisa seworth it ini... - review iphone 17
OUTPUT : kalau apple niat , iphone bisa seworth it ini . . . - review iphone 17
----------------------------------------------------------------------
INPUT  : xiaomi pun ngeluh soal fenomena ini...
OUTPUT : xiaomi pun ngeluh soal fenomena ini . . .
----------------------------------------------------------------------


---
## 5b. Stopword Removal

Menghapus **kata-kata umum (stopwords)** Bahasa Indonesia yang tidak membawa
makna topik spesifik (misal: *yang, dan, di, ini, itu, untuk, dengan, ke, dari, pada*).
Langkah ini membantu IndoBERT lebih fokus pada **kata bermuatan makna (content words)**
sehingga embedding lebih representatif terhadap topik judul.

In [9]:
# ============================================================
# BAGIAN 5b: STOPWORD REMOVAL
# ============================================================

# ─── Daftar stopword Bahasa Indonesia ────────────────────────────────────────
# Mencakup kata fungsi umum yang tidak membawa makna topik
INDONESIAN_STOPWORDS = {
    # Kata ganti & artikel
    'yang', 'ini', 'itu', 'dan', 'atau', 'di', 'ke', 'dari', 'pada', 'untuk',
    'dengan', 'adalah', 'ada', 'akan', 'sudah', 'telah', 'bisa', 'bukan',
    'tidak', 'tak', 'belum', 'jangan', 'agar', 'supaya', 'karena', 'sebab',
    'jika', 'bila', 'kalau', 'maka', 'ketika', 'saat', 'setelah', 'sebelum',
    'sehingga', 'walaupun', 'meskipun', 'bahwa', 'namun', 'tetapi', 'tapi',
    # Kata ganti orang
    'saya', 'aku', 'kamu', 'anda', 'dia', 'mereka', 'kita', 'kami', 'kamu',
    'kami', 'nya', 'ku', 'mu',
    # Kata keterangan umum
    'juga', 'lagi', 'masih', 'sudah', 'pun', 'pula', 'hanya', 'saja', 'lebih',
    'sangat', 'sekali', 'begitu', 'seperti', 'antara', 'oleh', 'bagi', 'tentang',
    'dalam', 'luar', 'atas', 'bawah', 'sejak', 'hingga', 'sampai', 'selama',
    'kemudian', 'lalu', 'jadi', 'menjadi', 'merupakan', 'yaitu', 'yakni',
    # Kata bantu umum
    'si', 'sang', 'para', 'se', 'tiap', 'setiap', 'semua', 'seluruh', 'beberapa',
    'paling', 'makin', 'semakin', 'cukup', 'agak', 'hampir',
    # Konjungsi
    'serta', 'maupun', 'baik', 'buat', 'tanpa', 'kecuali', 'bahkan',
    # Kata tanya
    'apa', 'siapa', 'dimana', 'kemana', 'darimana', 'kapan', 'berapa', 'mengapa',
    'kenapa', 'bagaimana', 'gimana',
    # Singkat / informal
    'lg', 'gak', 'ga', 'udah', 'udh', 'nih', 'dong', 'deh', 'sih', 'yuk',
    'yup', 'ya', 'iya', 'oke', 'ok', 'gitu', 'gini', 'tuh',
}


def remove_stopwords(text):
    """
    Menghapus stopword Bahasa Indonesia dari teks yang sudah di-tokenisasi.

    Catatan: fungsi ini dijalankan SETELAH tokenize_stanza sehingga
    batas token sudah bersih. Token dengan panjang < 2 karakter juga
    dihapus untuk mengurangi noise (simbol sisa tanda baca).

    Parameters
    ----------
    text : str – teks hasil tokenisasi Stanza.

    Returns
    -------
    str – teks setelah stopword dihapus.
    """
    if not isinstance(text, str) or not text.strip():
        return text

    tokens = text.split()
    filtered = [
        tok for tok in tokens
        if tok.lower() not in INDONESIAN_STOPWORDS and len(tok) >= 2
    ]

    # Fallback: jika semua token terhapus, kembalikan teks asli
    return ' '.join(filtered) if filtered else text


# ─── Terapkan stopword removal setelah tokenisasi ────────────────────────────
df['judul_final'] = df['judul_tokenized'].apply(remove_stopwords)


# ─── Tampilkan perbandingan ───────────────────────────────────────────────────
print('Contoh Hasil Stopword Removal:')
print('=' * 70)
for _, row in df[['judul_tokenized', 'judul_final']].head(5).iterrows():
    print(f'SEBELUM : {row["judul_tokenized"]}')
    print(f'SESUDAH : {row["judul_final"]}')
    print('-' * 70)

# Statistik token yang dihapus
avg_before = df['judul_tokenized'].apply(lambda x: len(str(x).split())).mean()
avg_after  = df['judul_final'].apply(lambda x: len(str(x).split())).mean()
print(f'\nRata-rata jumlah token:')
print(f'  Sebelum stopword removal : {avg_before:.1f} token')
print(f'  Sesudah stopword removal : {avg_after:.1f} token')
print(f'  Token tereduksi          : {avg_before - avg_after:.1f} token/judul')


Contoh Hasil Stopword Removal:
SEBELUM : unboxing iphone 17 pro palsu yang sangat mirip aslinya . . .
SESUDAH : unboxing iphone 17 pro palsu mirip aslinya
----------------------------------------------------------------------
SEBELUM : raja terakhir hp samsung !
SESUDAH : raja terakhir hp samsung
----------------------------------------------------------------------
SEBELUM : rp1.599 juta ! ketika oppo niat bikin hp murah . . .
SESUDAH : rp1.599 juta oppo niat bikin hp murah
----------------------------------------------------------------------
SEBELUM : kalau apple niat , iphone bisa seworth it ini . . . - review iphone 17
SESUDAH : apple niat iphone seworth it review iphone 17
----------------------------------------------------------------------
SEBELUM : xiaomi pun ngeluh soal fenomena ini . . .
SESUDAH : xiaomi ngeluh soal fenomena
----------------------------------------------------------------------

Rata-rata jumlah token:
  Sebelum stopword removal : 11.9 token
  Sesudah stopw

---
## 6. Load Model IndoBERT

Menggunakan model **`indolem/indobert-base-uncased`** dari Hugging Face.
Model ini dilatih khusus untuk Bahasa Indonesia.

In [10]:
# ============================================================
# BAGIAN 6: LOAD MODEL INDOBERT
# ============================================================

INDOBERT_MODEL_NAME = 'indolem/indobert-base-uncased'

print(f'Memuat model IndoBERT: {INDOBERT_MODEL_NAME}')
print('(Download pertama kali memerlukan beberapa menit)\n')

# ─── Load Tokenizer IndoBERT ──────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(INDOBERT_MODEL_NAME)
print(f'Tokenizer berhasil dimuat.')
print(f'  Vocab size  : {tokenizer.vocab_size:,}')

# ─── Load Model IndoBERT ──────────────────────────────────────
model = AutoModel.from_pretrained(INDOBERT_MODEL_NAME)
model = model.to(device)   # pindahkan ke GPU jika tersedia
model.eval()               # mode evaluasi (non-training)

print(f'\nModel berhasil dimuat → device: {device}')
print(f'  Hidden size : {model.config.hidden_size}')
print(f'  Num layers  : {model.config.num_hidden_layers}')
print(f'  Num heads   : {model.config.num_attention_heads}')

Memuat model IndoBERT: indolem/indobert-base-uncased
(Download pertama kali memerlukan beberapa menit)

Tokenizer berhasil dimuat.
  Vocab size  : 31,923


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 368.72it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: indolem/indobert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Model berhasil dimuat → device: cpu
  Hidden size : 768
  Num layers  : 12
  Num heads   : 12


---
## 7. Generate Embedding Judul Video

Setiap judul video diubah menjadi **embedding vector** menggunakan token **[CLS]**
dari output last hidden state IndoBERT sebagai representasi kalimat.

In [11]:
# ============================================================
# BAGIAN 7: GENERATE EMBEDDING JUDUL VIDEO
# (Attention-weighted Mean Pooling — lebih baik dari CLS token)
# ============================================================

def generate_all_embeddings(texts, tokenizer, model, device,
                            batch_size=32, max_length=128):
    """
    Menghasilkan sentence embedding menggunakan Attention-weighted Mean Pooling.

    Mengapa bukan [CLS]?
    --------------------
    Token [CLS] BERT dilatih untuk classification, bukan sentence similarity.
    Attention-weighted mean pooling merata-rata SEMUA token (kecuali padding)
    dengan bobot dari attention mask, sehingga representasi kalimat lebih kaya
    dan distribusi cosine similarity lebih tersebar (tidak semua ~0.85).

    Parameters
    ----------
    texts      : list of str – daftar teks judul yang sudah dipreproses.
    tokenizer  : IndoBERT tokenizer.
    model      : IndoBERT model.
    device     : torch.device – CPU atau GPU.
    batch_size : int – jumlah teks per batch.
    max_length : int – panjang token maksimum (default 128).

    Returns
    -------
    np.ndarray – shape (n_texts, hidden_size), yaitu (n, 768).
    """
    all_embeddings = []

    for start in tqdm(range(0, len(texts), batch_size),
                      desc='Generating embeddings', unit='batch'):
        batch_texts = texts[start: start + batch_size]

        # Tokenisasi seluruh batch sekaligus
        inputs = tokenizer(
            batch_texts,
            return_tensors='pt',
            max_length=max_length,
            truncation=True,
            padding=True
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Forward pass (tanpa gradient untuk hemat memori)
        with torch.no_grad():
            outputs = model(**inputs)

        # ── Attention-weighted Mean Pooling ──────────────────────────────────
        # token_embeddings : (batch_size, seq_len, 768)
        # attention_mask   : (batch_size, seq_len)  — 1=token asli, 0=padding
        token_embeddings = outputs.last_hidden_state          # (B, L, 768)
        attention_mask   = inputs['attention_mask']            # (B, L)

        # Perluas mask ke dimensi embedding: (B, L) → (B, L, 768)
        mask_expanded = attention_mask.unsqueeze(-1).expand(
            token_embeddings.size()
        ).float()

        # Jumlah embedding berbobot (token padding diabaikan)
        sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)  # (B, 768)

        # Jumlah token asli per sampel (clamp ≥ 1e-9 untuk hindari div-by-zero)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)           # (B, 768)

        # Mean pooling
        mean_embeddings = sum_embeddings / sum_mask                           # (B, 768)
        # ────────────────────────────────────────────────────────────────────

        all_embeddings.append(mean_embeddings.cpu().numpy())

    # Gabung semua batch → (n_texts, 768)
    return np.vstack(all_embeddings)


# ─── Jalankan Generate Embedding ─────────────────────────────
print(f'Memulai embedding untuk {len(df):,} judul video...')
print('Metode: Attention-weighted Mean Pooling (lebih baik dari [CLS])')
print('(Gunakan GPU untuk mempercepat proses)\n')

# Gunakan kolom judul_final (sudah melalui stopword removal)
texts_to_embed = df['judul_final'].tolist()

video_embeddings = generate_all_embeddings(
    texts=texts_to_embed,
    tokenizer=tokenizer,
    model=model,
    device=device,
    batch_size=32
)

# Simpan embedding ke DataFrame sebagai list per baris
df['embedding'] = list(video_embeddings)

print(f'\nEmbedding berhasil dibuat!')
print(f'  Shape matrix  : {video_embeddings.shape}')
print(f'  Jumlah video  : {video_embeddings.shape[0]:,}')
print(f'  Dimensi vektor: {video_embeddings.shape[1]}')
print(f'  Dtype         : {video_embeddings.dtype}')
print(f'  Metode        : Attention-weighted Mean Pooling')


Memulai embedding untuk 10,000 judul video...
Metode: Attention-weighted Mean Pooling (lebih baik dari [CLS])
(Gunakan GPU untuk mempercepat proses)



Generating embeddings: 100%|██████████| 313/313 [10:34<00:00,  2.03s/batch]


Embedding berhasil dibuat!
  Shape matrix  : (10000, 768)
  Jumlah video  : 10,000
  Dimensi vektor: 768
  Dtype         : float32
  Metode        : Attention-weighted Mean Pooling


---
## 8. Agregasi Vector Channel (Mean Pooling)

Setiap channel memiliki banyak video → semua embedding video dijumlah rata-rata
(**mean pooling**) sehingga menghasilkan **satu vektor representasi per channel**.

```
Channel A → [emb_vid1, emb_vid2, ..., emb_vidN]
          → mean(emb_vid1 ... emb_vidN)
          → 1 vektor channel (768-dim)
```

In [12]:
# ============================================================
# BAGIAN 8: CHANNEL VECTOR AGGREGATION (MEAN POOLING + L2 NORMALIZATION)
# ============================================================

from sklearn.preprocessing import normalize

def aggregate_channel_embeddings(df):
    """
    Mengagregasi embedding video menjadi satu vektor per channel
    menggunakan Mean Pooling, lalu menerapkan L2 Normalization.

    Mengapa L2 Normalization?
    -------------------------
    Setelah mean pooling, panjang (norm) vektor antar channel berbeda-beda.
    L2 normalization menjadikan semua vektor berunit norm = 1 (pada unit hypersphere),
    sehingga cosine similarity dihitung murni dari ARAH vektor (topik konten),
    bukan pengaruh panjang vektor. Hasilnya: distribusi similarity lebih tersebar
    dan diskriminatif.

    Parameters
    ----------
    df : DataFrame – harus memiliki kolom 'nama_channel' dan 'embedding'.

    Returns
    -------
    dict – { nama_channel: np.ndarray (768,) } — sudah L2-normalized.
    """
    channel_vectors = {}

    grouped = list(df.groupby('nama_channel'))

    for channel_name, group in tqdm(grouped, desc='Agregasi channel', unit='channel'):
        # Stack semua embedding video → (n_videos, 768)
        video_embs = np.stack(group['embedding'].values)

        # Mean Pooling → (768,)
        channel_vector = np.mean(video_embs, axis=0)

        channel_vectors[channel_name] = channel_vector

    return channel_vectors


# ─── Jalankan Agregasi ────────────────────────────────────────
channel_vectors = aggregate_channel_embeddings(df)

channel_names  = list(channel_vectors.keys())
channel_matrix = np.stack([channel_vectors[ch] for ch in channel_names])

# ── L2 Normalization ─────────────────────────────────────────────────────────
# Normalisasi setiap vektor channel menjadi unit vector (norm = 1)
# Efek: cosine similarity menjadi lebih diskriminatif (distribusi lebih tersebar)
channel_matrix = normalize(channel_matrix, norm='l2')   # (n_channels, 768)

# Update channel_vectors dengan versi yang sudah dinormalisasi
for i, ch in enumerate(channel_names):
    channel_vectors[ch] = channel_matrix[i]
# ─────────────────────────────────────────────────────────────────────────────

print(f'\nAgregasi selesai!')
print(f'  Jumlah channel  : {len(channel_names)}')
print(f'  Shape matrix    : {channel_matrix.shape}')
print(f'  L2 Normalization: ✓ (semua vektor ber-norm = 1.0)')

# Verifikasi norm
norms = np.linalg.norm(channel_matrix, axis=1)
print(f'  Norm (min-max)  : {norms.min():.4f} – {norms.max():.4f}  (seharusnya ~1.0)')

# Tampilkan 5 channel pertama
print('\nContoh vektor channel (5 nilai pertama dari vektor):')
print('=' * 65)
for ch in channel_names[:5]:
    vec     = channel_vectors[ch]
    n_video = len(df[df['nama_channel'] == ch])
    print(f'{ch:<40} | {n_video:>3} video | vec[:5]={vec[:5].round(4)}')


Agregasi channel:   0%|          | 0/100 [00:00<?, ?channel/s]

Agregasi channel: 100%|██████████| 100/100 [00:00<00:00, 1543.38channel/s]


Agregasi selesai!
  Jumlah channel  : 100
  Shape matrix    : (100, 768)
  L2 Normalization: ✓ (semua vektor ber-norm = 1.0)
  Norm (min-max)  : 1.0000 – 1.0000  (seharusnya ~1.0)

Contoh vektor channel (5 nilai pertama dari vektor):
@AfifYulistian                           | 100 video | vec[:5]=[-0.0158 -0.0157 -0.0812  0.044  -0.009 ]
@AlshadAhmad                             | 100 video | vec[:5]=[-0.0091 -0.0306 -0.0712  0.013  -0.0242]
@Anak.Kuliner                            | 100 video | vec[:5]=[-0.0091 -0.0315 -0.0733  0.0293 -0.0156]
@AquariusMusikindo                       | 100 video | vec[:5]=[-0.0142 -0.0082 -0.0708  0.0246 -0.0149]
@ArisSportTv                             | 100 video | vec[:5]=[-0.0161 -0.0146 -0.0752  0.0128 -0.0284]


---
## 9. Cosine Similarity Matrix

Menghitung kemiripan antar channel menggunakan **Cosine Similarity**.
Hasilnya berupa matriks simetris berukuran **(n_channel × n_channel)**.

In [13]:
# ============================================================
# BAGIAN 9: COSINE SIMILARITY MATRIX
# ============================================================

# Hitung Cosine Similarity antar semua pasangan channel
# Input : channel_matrix  shape (n_channels, 768)
# Output: similarity_matrix shape (n_channels, n_channels)
similarity_matrix = cosine_similarity(channel_matrix)

# Konversi ke DataFrame agar mudah diakses via nama channel
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=channel_names,
    columns=channel_names
)

print('Cosine Similarity Matrix berhasil dibuat!')
print(f'  Shape : {similarity_matrix.shape}')
print(f'  (Baris & Kolom = {len(channel_names)} channel)')

# Statistik nilai similarity non-diagonal (antar channel berbeda)
mask     = ~np.eye(len(channel_names), dtype=bool)
off_diag = similarity_matrix[mask]

print(f'\nStatistik Cosine Similarity (antar channel berbeda):')
print(f'  Min  : {off_diag.min():.4f}')
print(f'  Max  : {off_diag.max():.4f}')
print(f'  Mean : {off_diag.mean():.4f}')
print(f'  Std  : {off_diag.std():.4f}')

# Tampilkan sebagian matriks (5×5)
print('\nCuplikan Similarity Matrix (5×5):')
similarity_df.iloc[:5, :5].round(4)

Cosine Similarity Matrix berhasil dibuat!
  Shape : (100, 100)
  (Baris & Kolom = 100 channel)

Statistik Cosine Similarity (antar channel berbeda):
  Min  : 0.6114
  Max  : 0.9908
  Mean : 0.9174
  Std  : 0.0472

Cuplikan Similarity Matrix (5×5):


,@AfifYulistian,@AlshadAhmad,@Anak.Kuliner,@AquariusMusikindo,@ArisSportTv
@AfifYulistian,1.0000,0.9556,0.9523,0.9334,0.9063
@AlshadAhmad,0.9556,1.0000,0.9645,0.9224,0.9440
@Anak.Kuliner,0.9523,0.9645,1.0000,0.9115,0.9166
@AquariusMusikindo,0.9334,0.9224,0.9115,1.0000,0.9100
@ArisSportTv,0.9063,0.9440,0.9166,0.9100,1.0000


---
## 10. Fungsi Rekomendasi

Fungsi `recommend_channel(channel_name, top_k)` mengembalikan **Top-K channel paling mirip**
berdasarkan Cosine Similarity dari vektor representasi channel.

In [14]:
# ============================================================
# BAGIAN 10: FUNGSI REKOMENDASI
# ============================================================

def recommend_channel(channel_name, top_k=5):
    """
    Merekomendasikan Top-K channel YouTube paling mirip dengan channel
    yang diberikan, berdasarkan Cosine Similarity embedding IndoBERT.

    Alur:
    1. Ambil baris similarity channel input dari similarity_df.
    2. Hilangkan channel itu sendiri.
    3. Urutkan berdasarkan similarity tertinggi.
    4. Kembalikan Top-K channel beserta info pendukung.

    Parameters
    ----------
    channel_name : str – nama channel input, misalnya '@GadgetIn'.
    top_k        : int – jumlah rekomendasi yang dikembalikan.

    Returns
    -------
    pd.DataFrame – kolom: rank, nama_channel, kategori,
                   jumlah_pelanggan, similarity_score.
    """
    # Gunakan variabel global similarity_df dan df
    global similarity_df, df

    if channel_name not in similarity_df.index:
        available = list(similarity_df.index)
        raise ValueError(
            f"Channel '{channel_name}' tidak ditemukan.\n"
            f"Channel tersedia: {available}"
        )

    # Ambil skor similarity, hapus diri sendiri
    sim_scores = similarity_df.loc[channel_name].drop(labels=channel_name)

    # Urutkan similarity tertinggi → ambil Top-K
    top_k_channels = sim_scores.sort_values(ascending=False).head(top_k)

    # Info pendukung dari dataset asli
    channel_info = (
        df.drop_duplicates(subset='nama_channel')
          .set_index('nama_channel')[['kategori', 'jumlah_pelanggan']]
    )

    results = []
    for rank, (ch, score) in enumerate(top_k_channels.items(), start=1):
        kategori  = channel_info.loc[ch, 'kategori']         if ch in channel_info.index else '-'
        pelanggan = channel_info.loc[ch, 'jumlah_pelanggan'] if ch in channel_info.index else 0
        results.append({
            'rank'             : rank,
            'nama_channel'     : ch,
            'kategori'         : kategori,
            'jumlah_pelanggan' : pelanggan,
            'similarity_score' : round(float(score), 4)
        })

    return pd.DataFrame(results).set_index('rank')


# ─── Contoh Penggunaan ────────────────────────────────────────
print('=' * 65)
print('           CONTOH REKOMENDASI CHANNEL')
print('=' * 65)

input_channel = channel_names[0]  # channel pertama dalam dataset
TOP_K = 5

print(f'Channel Input : {input_channel}')
print(f'Top-K         : {TOP_K}')
print('-' * 65)

rec_df = recommend_channel(input_channel, top_k=TOP_K)

print(f'\nTop-{TOP_K} Channel yang Direkomendasikan untuk "{input_channel}":\n')
for idx, row in rec_df.iterrows():
    print(f'  {idx}. {row["nama_channel"]:<40} (similarity: {row["similarity_score"]:.4f})')

print()
print(rec_df.to_string())

           CONTOH REKOMENDASI CHANNEL
Channel Input : @AfifYulistian
Top-K         : 5
-----------------------------------------------------------------

Top-5 Channel yang Direkomendasikan untuk "@AfifYulistian":

  1. @Miawaug                                 (similarity: 0.9860)
  2. @MILYHYA                                 (similarity: 0.9696)
  3. @GarasiDrift                             (similarity: 0.9641)
  4. @HujanTandaTanya                         (similarity: 0.9633)
  5. @LuckyHakimChannel                       (similarity: 0.9631)

            nama_channel    kategori  jumlah_pelanggan  similarity_score
rank                                                                    
1               @Miawaug      Gaming          24900000            0.9860
2               @MILYHYA      Gaming           5890000            0.9696
3           @GarasiDrift  Automotive           2160000            0.9641
4       @HujanTandaTanya   Education            594000            0.9633
5     @Luck

In [15]:
# ─── Uji Coba dengan 3 Channel Pertama ───────────────────────
test_channels = channel_names[:3]

for ch in test_channels:
    print('=' * 65)
    print(f' Rekomendasi untuk: {ch}')
    print('=' * 65)
    rec = recommend_channel(ch, top_k=5)
    for i, row in rec.iterrows():
        print(f'  {i}. {row["nama_channel"]:<40} ({row["similarity_score"]:.4f})')
    print()

 Rekomendasi untuk: @AfifYulistian
  1. @Miawaug                                 (0.9860)
  2. @MILYHYA                                 (0.9696)
  3. @GarasiDrift                             (0.9641)
  4. @HujanTandaTanya                         (0.9633)
  5. @LuckyHakimChannel                       (0.9631)

 Rekomendasi untuk: @AlshadAhmad
  1. @letdahyper                              (0.9755)
  2. @BeemzAryo                               (0.9752)
  3. @farida.nurhan                           (0.9709)
  4. @HujanTandaTanya                         (0.9708)
  5. @PANJIPETUALANG_REAL                     (0.9693)

 Rekomendasi untuk: @Anak.Kuliner
  1. @KUBILER                                 (0.9851)
  2. @MamankKuliner                           (0.9838)
  3. @Kenandgrat                              (0.9823)
  4. @tanboykun                               (0.9806)
  5. @BoengkoesNetwork                        (0.9794)



---
## 10b. Fungsi Rekomendasi Berdasarkan Input Judul Video

Fungsi `recommend_by_title(judul_input, top_k, top_judul)` menerima **judul video sembarang**
sebagai input (bukan nama channel), lalu:

1. Menerapkan **pipeline preprocessing yang sama** (text_cleaning → case_folding → tokenize_stanza)
2. Men-*generate* **embedding IndoBERT** untuk judul input tersebut
3. Menghitung **cosine similarity** antara embedding judul input vs. setiap *channel vector* → dapatkan **Top-K channel paling mirip**
4. Untuk setiap channel yang direkomendasikan, menghitung similarity embedding judul input vs. **setiap video** di channel tersebut → dapatkan **Top-N judul video paling mirip**
5. Mengembalikan DataFrame lengkap: **nama channel + skor similarity channel + daftar judul mirip**

**Keunggulan:** Pengguna tidak hanya tahu *channel mana* yang mirip, tetapi juga
*judul-judul spesifik* dari channel tersebut yang paling relevan dengan topik yang mereka cari.

In [16]:
# ============================================================
# BAGIAN 10b: FUNGSI REKOMENDASI BERDASARKAN INPUT JUDUL
# ============================================================

def preprocess_single_title(judul_input):
    """
    Menerapkan pipeline preprocessing yang sama seperti data training
    pada satu judul video input:
      1. text_cleaning
      2. case_folding
      3. tokenize_stanza

    Parameters
    ----------
    judul_input : str – judul video sembarang dari pengguna.

    Returns
    -------
    str – teks yang sudah dipreproses, siap di-embed oleh IndoBERT.
    """
    cleaned   = text_cleaning(judul_input)
    lowered   = case_folding(cleaned)
    tokenized = tokenize_stanza(lowered)
    return tokenized


def get_title_embedding(processed_text):
    """
    Men-generate embedding IndoBERT [CLS] untuk satu teks yang sudah dipreproses.

    Parameters
    ----------
    processed_text : str – teks hasil preprocess_single_title.

    Returns
    -------
    np.ndarray – vektor embedding shape (768,).
    """
    global tokenizer, model, device

    inputs = tokenizer(
        processed_text,
        return_tensors='pt',
        max_length=128,
        truncation=True,
        padding=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    # Token [CLS] → indeks 0 pada sequence dimension
    # Shape: (1, seq_len, 768) → [:, 0, :] → (1, 768) → [0] → (768,)
    embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()[0]
    return embedding


def get_similar_titles_in_channel(title_embedding, channel_name, top_n=3):
    """
    Mencari top-N judul video dalam satu channel yang paling mirip
    dengan embedding judul input.

    Parameters
    ----------
    title_embedding : np.ndarray – embedding judul input shape (768,).
    channel_name    : str         – nama channel yang dicari.
    top_n           : int         – jumlah judul mirip yang dikembalikan.

    Returns
    -------
    list of dict – [{judul, similarity_judul}, ...] diurutkan similarity turun.
    """
    global df

    # Ambil semua video dari channel ini beserta embedding-nya
    channel_videos = df[df['nama_channel'] == channel_name].copy()

    if channel_videos.empty:
        return []

    # Stack semua embedding video channel → (n_videos, 768)
    video_embs = np.stack(channel_videos['embedding'].values)

    # Cosine similarity: input embedding vs. semua video di channel ini
    sims = cosine_similarity(
        title_embedding.reshape(1, -1),   # (1, 768)
        video_embs                         # (n_videos, 768)
    )[0]  # → (n_videos,)

    # Ambil Top-N indeks
    top_n_idx = np.argsort(sims)[::-1][:top_n]

    judul_list = channel_videos['judul'].values
    result = [
        {
            'judul'            : judul_list[i],
            'similarity_judul' : round(float(sims[i]), 4)
        }
        for i in top_n_idx
    ]
    return result


def recommend_by_title(judul_input, top_k=5, top_judul=3, verbose=True):
    """
    Merekomendasikan Top-K channel YouTube yang paling mirip dengan judul
    video input, sekaligus menampilkan top_judul judul video dari masing-masing
    channel yang paling relevan dengan input.

    Alur:
    1. Preprocessing judul input (cleaning → case folding → tokenisasi Stanza).
    2. Generate embedding IndoBERT [CLS] untuk judul input.
    3. Hitung cosine similarity: embedding input vs. channel_matrix
       → dapatkan Top-K channel paling mirip.
    4. Untuk setiap channel: hitung similarity embedding input vs.
       setiap video di channel tersebut → dapatkan top_judul judul terdekat.
    5. Kembalikan DataFrame hasil rekomendasi (channel + judul mirip).

    Parameters
    ----------
    judul_input : str  – judul video sembarang (masukan pengguna).
                         Contoh: "Review HP gaming terbaik 2024"
    top_k       : int  – jumlah channel yang direkomendasikan (default 5).
    top_judul   : int  – jumlah judul mirip per channel yang ditampilkan (default 3).
    verbose     : bool – jika True, tampilkan langkah preprocessing ke layar.

    Returns
    -------
    pd.DataFrame – kolom: nama_channel, kategori, jumlah_pelanggan,
                   similarity_channel, judul_mirip (list of dict).
    """
    global channel_names, channel_matrix, df

    # ── Langkah 1: Preprocessing ─────────────────────────────────
    if verbose:
        print(f'[1/4] Preprocessing judul input...')
        print(f'      Input    : {judul_input}')

    processed = preprocess_single_title(judul_input)

    if verbose:
        print(f'      Processed: {processed}')

    # ── Langkah 2: Generate Embedding ────────────────────────────
    if verbose:
        print(f'[2/4] Generate embedding IndoBERT...')

    title_embedding = get_title_embedding(processed)   # shape (768,)

    # ── Langkah 3: Cosine Similarity vs. Channel Matrix ──────────
    if verbose:
        print(f'[3/4] Cosine similarity judul vs. {len(channel_names)} channel vector...')

    sim_scores = cosine_similarity(
        title_embedding.reshape(1, -1),   # (1, 768)
        channel_matrix                     # (n_channels, 768)
    )[0]  # → (n_channels,)

    top_k_idx = np.argsort(sim_scores)[::-1][:top_k]

    # ── Langkah 4: Cari judul mirip per channel ──────────────────
    if verbose:
        print(f'[4/4] Mencari top-{top_judul} judul mirip per channel...')

    channel_info = (
        df.drop_duplicates(subset='nama_channel')
          .set_index('nama_channel')[['kategori', 'jumlah_pelanggan']]
    )

    results = []
    for rank, idx in enumerate(top_k_idx, start=1):
        ch_name   = channel_names[idx]
        ch_score  = float(sim_scores[idx])
        kategori  = channel_info.loc[ch_name, 'kategori']         if ch_name in channel_info.index else '-'
        pelanggan = channel_info.loc[ch_name, 'jumlah_pelanggan'] if ch_name in channel_info.index else 0

        # Judul mirip dari channel ini
        judul_mirip = get_similar_titles_in_channel(
            title_embedding, ch_name, top_n=top_judul
        )

        results.append({
            'rank'               : rank,
            'nama_channel'       : ch_name,
            'kategori'           : kategori,
            'jumlah_pelanggan'   : pelanggan,
            'similarity_channel' : round(ch_score, 4),
            'judul_mirip'        : judul_mirip        # list of {judul, similarity_judul}
        })

    return pd.DataFrame(results).set_index('rank')


def print_rekomendasi_judul(judul_input, top_k=5, top_judul=3):
    """
    Wrapper cetak hasil recommend_by_title dengan format yang rapi dan lengkap:
    menampilkan nama channel, kategori, similarity channel, dan
    daftar judul mirip beserta similarity-nya masing-masing.

    Parameters
    ----------
    judul_input : str – judul video input dari pengguna.
    top_k       : int – jumlah channel yang direkomendasikan.
    top_judul   : int – jumlah judul mirip yang ditampilkan per channel.
    """
    print('=' * 70)
    print(f'  INPUT JUDUL : "{judul_input}"')
    print('=' * 70)

    rec = recommend_by_title(judul_input, top_k=top_k, top_judul=top_judul, verbose=True)

    print(f'\nTop-{top_k} Channel Direkomendasikan + Judul Paling Mirip:\n')
    print('-' * 70)

    for rank, row in rec.iterrows():
        print(f'  #{rank}. {row["nama_channel"]:<38} [{row["kategori"]}]')
        print(f'      Similarity Channel : {row["similarity_channel"]:.4f}')
        print(f'      Judul Mirip dari Channel Ini:')
        for j, item in enumerate(row['judul_mirip'], start=1):
            print(f'        {j}. "{item["judul"]}"')
            print(f'           (similarity judul: {item["similarity_judul"]:.4f})')
        print('-' * 70)

    print()
    return rec


# ─── Demo: Contoh Penggunaan ──────────────────────────────────────────────────
demo_titles = [
    'Review HP gaming terbaik 2024',
    'Resep masakan ayam kecap sederhana',
    'Berita politik terkini hari ini',
    'Highlights pertandingan sepak bola liga Indonesia',
]

for judul in demo_titles:
    rec_demo = print_rekomendasi_judul(judul, top_k=5, top_judul=3)
    print()


  INPUT JUDUL : "Review HP gaming terbaik 2024"
[1/4] Preprocessing judul input...
      Input    : Review HP gaming terbaik 2024
      Processed: review hp gaming terbaik 2024
[2/4] Generate embedding IndoBERT...
[3/4] Cosine similarity judul vs. 100 channel vector...
[4/4] Mencari top-3 judul mirip per channel...

Top-5 Channel Direkomendasikan + Judul Paling Mirip:

----------------------------------------------------------------------
  #1. @PSSITV                                [Sports]
      Similarity Channel : 0.5576
      Judul Mirip dari Channel Ini:
        1. "Post-Match Press Conference: Bahrain vs Indonesia"
           (similarity judul: 0.6103)
        2. "PSSI National Coaching Conference 2025"
           (similarity judul: 0.5768)
        3. "Freeport Grassroots Tournament 2025"
           (similarity judul: 0.5470)
----------------------------------------------------------------------
  #2. @GadgetGaul                            [Gadgets]
      Similarity Channel : 0.

---
## 11. Evaluasi Sistem: Precision@K

**Precision@K** mengukur proporsi rekomendasi yang relevan dalam Top-K hasil rekomendasi.

$$\text{Precision@K} = \frac{\text{Jumlah rekomendasi relevan}}{K}$$

**Definisi relevansi:** Channel dianggap relevan jika memiliki **kategori yang sama**
dengan channel input (ground truth berbasis kategori).

In [17]:
# ============================================================
# BAGIAN 11: EVALUASI – PRECISION@K
# ============================================================

# ─── Mapping channel → kategori ──────────────────────────────
channel_category_map = (
    df.drop_duplicates(subset='nama_channel')
      .set_index('nama_channel')['kategori']
      .to_dict()
)


def precision_at_k(channel_name, k):
    """
    Menghitung Precision@K untuk satu channel.

    Definisi Relevansi:
        Channel dikatakan relevan jika memiliki kategori yang sama
        dengan channel input (ground truth berbasis kategori).

    Formula:
        Precision@K = |{ch relevan dalam Top-K}| / K

    Parameters
    ----------
    channel_name : str – channel yang dievaluasi.
    k            : int – jumlah rekomendasi yang dievaluasi.

    Returns
    -------
    float – nilai Precision@K dalam rentang [0.0, 1.0].
    """
    global similarity_df, channel_category_map

    if channel_name not in similarity_df.index:
        return 0.0

    target_category = channel_category_map.get(channel_name)
    if target_category is None:
        return 0.0

    # Dapatkan Top-K channel yang direkomendasikan
    sim_scores     = similarity_df.loc[channel_name].drop(labels=channel_name)
    top_k_channels = sim_scores.sort_values(ascending=False).head(k).index.tolist()

    # Hitung jumlah yang relevan (kategori sama)
    relevant_count = sum(
        1 for ch in top_k_channels
        if channel_category_map.get(ch) == target_category
    )

    return relevant_count / k


def evaluate_system(k_values=None):
    """
    Mengevaluasi keseluruhan sistem rekomendasi menggunakan Precision@K
    untuk berbagai nilai K, dirata-rata di seluruh channel.

    Parameters
    ----------
    k_values : list of int – nilai K yang dievaluasi.

    Returns
    -------
    pd.DataFrame – Precision@K per channel beserta baris rata-rata.
    """
    global similarity_df, channel_category_map

    if k_values is None:
        k_values = [1, 3, 5, 10]

    all_channels = list(similarity_df.index)
    eval_results = []

    for ch in tqdm(all_channels, desc='Evaluasi Precision@K', unit='channel'):
        row = {'channel': ch, 'kategori': channel_category_map.get(ch, '-')}
        for k in k_values:
            row[f'P@{k}'] = round(precision_at_k(ch, k), 4)
        eval_results.append(row)

    eval_df = pd.DataFrame(eval_results).set_index('channel')

    # ─── Tambahkan baris rata-rata ────────────────────────────
    avg_values = {'kategori': 'AVERAGE'}
    for k in k_values:
        avg_values[f'P@{k}'] = round(eval_df[f'P@{k}'].mean(), 4)

    avg_df = pd.DataFrame([avg_values], index=['--- AVERAGE ---'])
    eval_df = pd.concat([eval_df, avg_df])

    return eval_df


# ─── Jalankan Evaluasi ────────────────────────────────────────
K_VALUES = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

print('Menjalankan evaluasi Precision@K untuk semua channel...')
eval_df = evaluate_system(k_values=K_VALUES)

print('\n' + '=' * 65)
print('         HASIL EVALUASI SISTEM REKOMENDASI')
print('=' * 65)
print(eval_df.to_string())
print('=' * 65)

# ─── Ringkasan rata-rata ──────────────────────────────────────
print('\nRingkasan Rata-Rata Precision@K:')
avg_series = eval_df.loc['--- AVERAGE ---']
for k in K_VALUES:
    print(f'  Precision@{k:>2} = {avg_series[f"P@{k}"]:.4f}')

Menjalankan evaluasi Precision@K untuk semua channel...


Evaluasi Precision@K: 100%|██████████| 100/100 [00:00<00:00, 171.92channel/s]


         HASIL EVALUASI SISTEM REKOMENDASI
                                      kategori   P@1   P@2     P@3     P@4  P@5     P@6     P@7     P@8     P@9  P@10
@AfifYulistian                          Gaming  1.00  1.00  0.6667  0.5000  0.4  0.5000  0.4286  0.3750  0.4444  0.40
@AlshadAhmad                           Animals  0.00  0.50  0.3333  0.2500  0.4  0.3333  0.4286  0.3750  0.3333  0.30
@Anak.Kuliner                             Food  1.00  1.00  1.0000  1.0000  1.0  0.8333  0.7143  0.7500  0.6667  0.60
@AquariusMusikindo                       Music  1.00  1.00  1.0000  1.0000  1.0  1.0000  1.0000  1.0000  0.8889  0.80
@ArisSportTv                            Sports  0.00  0.00  0.0000  0.0000  0.0  0.0000  0.0000  0.0000  0.0000  0.00
@AttaHalilintar                  Entertainment  1.00  1.00  0.6667  0.5000  0.6  0.6667  0.7143  0.6250  0.5556  0.50
@Audrey-A                              Animals  0.00  0.50  0.6667  0.5000  0.6  0.5000  0.4286  0.3750  0.3333  0.30
@AutonetMagz

In [18]:
# ─── Evaluasi Detail per Kategori ────────────────────────────
print('\nPrecision@K Per Kategori (rata-rata per kategori):')
print('=' * 55)

# Filter baris rata-rata
eval_df_clean = eval_df[eval_df['kategori'] != 'AVERAGE'].copy()

pk_cols = [f'P@{k}' for k in K_VALUES]

cat_summary = (
    eval_df_clean.groupby('kategori')[pk_cols]
    .mean()
    .round(4)
    .sort_values('P@5', ascending=False)
)
print(cat_summary.to_string())


Precision@K Per Kategori (rata-rata per kategori):
               P@1   P@2     P@3    P@4   P@5     P@6     P@7     P@8     P@9  P@10
kategori                                                                           
Gadgets        1.0  1.00  1.0000  0.975  0.96  0.9500  0.9429  0.8375  0.8111  0.73
News           0.9  0.85  0.8667  0.875  0.84  0.8333  0.8143  0.7625  0.6889  0.64
Music          0.9  0.90  0.8667  0.850  0.80  0.7667  0.7714  0.7375  0.6889  0.65
Food           0.7  0.65  0.6000  0.625  0.64  0.5666  0.5000  0.4625  0.4222  0.38
Entertainment  0.7  0.75  0.6667  0.625  0.58  0.5167  0.5000  0.4625  0.4222  0.42
Automotive     0.9  0.70  0.6667  0.575  0.52  0.4667  0.4572  0.4250  0.3778  0.34
Gaming         0.8  0.70  0.6000  0.550  0.46  0.4500  0.4286  0.4000  0.4000  0.36
Sports         0.6  0.55  0.5000  0.450  0.42  0.3833  0.3571  0.3250  0.3333  0.31
Education      0.7  0.70  0.6333  0.475  0.40  0.3833  0.3857  0.3375  0.3222  0.29
Animals        0.6  0.50

---
## 12. Simpan Hasil (Opsional)

Menyimpan semua artefak ke folder `output/` untuk penggunaan ulang.

In [19]:
# ============================================================
# BAGIAN OPSIONAL: SIMPAN HASIL EMBEDDING & SIMILARITY
# ============================================================

OUTPUT_DIR = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Simpan channel matrix dan nama channel
np.save(f'{OUTPUT_DIR}/channel_matrix.npy', channel_matrix)
pd.DataFrame({'nama_channel': channel_names}).to_csv(
    f'{OUTPUT_DIR}/channel_names.csv', index=False
)

# 2. Simpan similarity matrix
np.save(f'{OUTPUT_DIR}/similarity_matrix.npy', similarity_matrix)
similarity_df.to_csv(f'{OUTPUT_DIR}/similarity_matrix.csv')

# 3. Simpan hasil evaluasi
eval_df.to_csv(f'{OUTPUT_DIR}/evaluation_results.csv')

# 4. Simpan DataFrame hasil preprocessing
cols_to_save = ['id', 'nama_channel', 'kategori', 'judul',
                'judul_clean', 'judul_lower', 'judul_tokenized']
df[cols_to_save].to_csv(f'{OUTPUT_DIR}/preprocessed_data.csv', index=False)

print(f'Semua hasil disimpan ke folder: {OUTPUT_DIR}/')
for fname in os.listdir(OUTPUT_DIR):
    fsize = os.path.getsize(f'{OUTPUT_DIR}/{fname}') / 1024
    print(f'  {fname:<40} ({fsize:,.1f} KB)')
# ─── Tambahan untuk Flask App ────────────────────────────────
# Simpan video embeddings (dibutuhkan oleh web app)
np.save(f'{OUTPUT_DIR}/video_embeddings.npy', video_embeddings)

# Simpan metadata video lengkap (untuk ditampilkan di web app)
cols_meta = ['id', 'nama_channel', 'kategori', 'judul',
             'link', 'link_channel', 'jumlah_tayangan',
             'tanggal_upload', 'jumlah_pelanggan']
df[cols_meta].to_csv(f'{OUTPUT_DIR}/video_metadata.csv', index=False)

print(f'  video_embeddings.npy               ({os.path.getsize(f"{OUTPUT_DIR}/video_embeddings.npy")/1024:,.1f} KB)')
print(f'  video_metadata.csv                 ({os.path.getsize(f"{OUTPUT_DIR}/video_metadata.csv")/1024:,.1f} KB)')


Semua hasil disimpan ke folder: output/
  channel_matrix.npy                       (300.1 KB)
  channel_names.csv                        (1.5 KB)
  evaluation_results.csv                   (7.2 KB)
  preprocessed_data.csv                    (2,787.7 KB)
  similarity_matrix.csv                    (103.8 KB)
  similarity_matrix.npy                    (39.2 KB)
  video_embeddings.npy               (30,000.1 KB)
  video_metadata.csv                 (2,056.8 KB)


---
## Ringkasan Metodologi

| Tahap | Metode | Library |
|---|---|---|
| Text Cleaning | Hapus emoji, simbol non-standar, encoding rusak | `re` |
| Case Folding | Lowercase seluruh teks | built-in |
| Tokenisasi | Stanza Bahasa Indonesia | `stanza` |
| Sentence Embedding | IndoBERT token [CLS] | `transformers`, `torch` |
| Channel Representation | Mean Pooling semua video | `numpy` |
| Similarity | Cosine Similarity | `sklearn` |
| Rekomendasi | Top-K similarity tertinggi | `pandas`, `numpy` |
| Evaluasi | Precision@K (ground truth: kategori) | `numpy`, `pandas` |

**Model NLP:** `indolem/indobert-base-uncased`  
**Pendekatan:** Content-Based Filtering  
**Fitur Utama:** Judul video YouTube